In [0]:
#Make sure these tables exist:

spark.sql("SHOW TABLES IN workspace.ecommerce_lakehouse").display()


In [0]:
# Load Bronze Table
bronze_df = spark.read.table(
    "workspace.ecommerce_lakehouse.bronze_clickstream_events"
)

display(bronze_df)

In [0]:
#Convert Timestamp Currently ts is Unix time. Convert it to real timestamp.
from pyspark.sql.functions import col, to_timestamp, from_unixtime

silver_df = bronze_df.withColumn(
    "event_time",
    to_timestamp(from_unixtime(col("ts")))
)
# Drop raw timestamp:
silver_df = silver_df.drop("ts")

In [0]:
# Standardize Event Type, Normalize event names.
from pyspark.sql.functions import lower

silver_df = silver_df.withColumn(
    "event_type",
    lower(col("event_type"))
)

In [0]:
#Remove Duplicate Events. Real systems must remove duplicates.
silver_df = silver_df.dropDuplicates(
    ["user_id", "session_id", "product_id", "event_time"]
)

silver_events_clean

In [0]:
# Save Clean Silver Events
silver_df.write.format("delta") \
.mode("overwrite") \
.saveAsTable("workspace.ecommerce_lakehouse.silver_events_clean")

In [0]:
# Verify
display(
    spark.read.table(
        "workspace.ecommerce_lakehouse.silver_events_clean"
    )
)

## **Sessionization Algorithm**

In [0]:
# Define Window
from pyspark.sql.window import Window
from pyspark.sql.functions import lag
window_user = Window.partitionBy("user_id").orderBy("event_time")

In [0]:
# Calculate Time Gap
from pyspark.sql.functions import unix_timestamp

session_df = silver_df.withColumn(
    "prev_event_time",
    lag("event_time").over(window_user)
)
#Calculate gap:
session_df = session_df.withColumn(
    "time_gap",
    unix_timestamp("event_time") -
    unix_timestamp("prev_event_time")
)

In [0]:
# Detect New Session
from pyspark.sql.functions import when

session_df = session_df.withColumn(
    "new_session",
    when(col("time_gap") > 1800, 1).otherwise(0)
)

In [0]:
# Generate Session IDs
from pyspark.sql.functions import sum as spark_sum

session_df = session_df.withColumn(
    "session_number",
    spark_sum("new_session").over(window_user)
)

silver_sessionized_events

In [0]:
# Save Sessionized Events
session_df.write.format("delta") \
.mode("overwrite") \
.saveAsTable("workspace.ecommerce_lakehouse.silver_sessionized_events")
# Check:
display(
spark.read.table(
"workspace.ecommerce_lakehouse.silver_sessionized_events"
))

## **Create Behavioral Features **

In [0]:
# Now we generate ML-ready features.
# Feature Aggregration
from pyspark.sql.functions import count, avg

features_df = session_df.groupBy(
"user_id",
"session_number"
).agg(
count("*").alias("total_events"),
avg("price").alias("avg_price"),
count(
when(col("event_type") == "view", True)
).alias("views"),
count(
when(col("event_type") == "add_to_cart", True)
).alias("add_to_cart"),
count(
when(col("event_type") == "purchase", True)
).alias("purchases")
)

silver_user_behavior_features

In [0]:
# Save Feature Table
features_df.write.format("delta") \
.mode("overwrite") \
.saveAsTable("workspace.ecommerce_lakehouse.silver_user_behavior_features")

In [0]:
# Verify:
display(
spark.read.table(
"workspace.ecommerce_lakehouse.silver_user_behavior_features"
))

## **Product Popularity Windows**

In [0]:
# Load sessionized events
session_df = spark.read.table(
"workspace.ecommerce_lakehouse.silver_sessionized_events"
)

In [0]:
# Compute Product Popularity Windows, This calculates events per product per hour.
from pyspark.sql.functions import window, count

product_popularity = session_df.groupBy(
"product_id",
window("event_time", "1 hour")
).agg(
count("*").alias("events_last_hour")
)

In [0]:
# Save Product Popularity Table,Now you have time-window product demand signals.
product_popularity.write.format("delta") \
.mode("overwrite") \
.saveAsTable(
"workspace.ecommerce_lakehouse.silver_product_popularity_features"
)
#Verify:
display(
spark.read.table(
"workspace.ecommerce_lakehouse.silver_product_popularity_features"
)
)

## **RFM Features (Recency Frequency Monetary)**

In [0]:
#RFM metrics:
# Recency   → how recently user interacted
# Frequency → how many events user generated
# Monetary  → total purchase value
# These are very powerful ML features.

# Import Functions
from pyspark.sql.functions import max, count, sum, datediff, current_date


In [0]:
#Compute RFM Metrics
rfm_features = session_df.groupBy("user_id").agg(

max("event_time").alias("last_event_time"),

count("*").alias("frequency"),

sum("price").alias("monetary_value")

)

In [0]:
#Calculate Recency
rfm_features = rfm_features.withColumn(
"recency_days",
datediff(current_date(), "last_event_time")
)

In [0]:
#Save RFM Table
rfm_features.write.format("delta") \
.mode("overwrite") \
.saveAsTable(
"workspace.ecommerce_lakehouse.silver_user_rfm_features"
)

#Verify:
display(
spark.read.table(
"workspace.ecommerce_lakehouse.silver_user_rfm_features"
)
)